# GPT-OSS Reasoning Parser Smoke Test

This notebook is for quickly checking that GPT-OSS style outputs are cleaned correctly:

- leading `analysis` is removed from the reasoning text
- trailing `assistantfinal` is removed from the final message / JSON
- the same helper still works for DeepSeek-style `<think>...</think>` outputs


In [ ]:
from pathlib import Path
import json
import sys
from pprint import pprint

REPO_ROOT = Path("/playpen-ssd/smerrill/deception2")
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from reasoning_parser import (
    extract_reasoning_trace,
    reasoning_close_span_from_text,
    strip_reasoning_trace,
)


def balanced_json_candidates(text: str):
    spans = []
    depth = 0
    start = None
    in_str = False
    escaped = False

    for i, ch in enumerate(text):
        if in_str:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_str = False
            continue

        if ch == '"':
            in_str = True
            continue
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
            continue
        if ch == "}" and depth > 0:
            depth -= 1
            if depth == 0 and start is not None:
                spans.append((start, i + 1))
                start = None

    return [text[s:e] for s, e in spans]


def parse_clean_output(raw_text: str, model_name: str):
    reasoning = extract_reasoning_trace(raw_text, model_name=model_name)
    cleaned_final = strip_reasoning_trace(raw_text, model_name=model_name)
    parsed = None
    parse_error = None

    for candidate in reversed(balanced_json_candidates(cleaned_final)):
        try:
            parsed = json.loads(candidate)
            break
        except Exception as exc:
            parse_error = str(exc)

    if parsed is None:
        try:
            parsed = json.loads(cleaned_final)
        except Exception as exc:
            parse_error = str(exc)

    return {
        "model_name": model_name,
        "close_span": reasoning_close_span_from_text(raw_text, model_name=model_name),
        "reasoning": reasoning,
        "cleaned_final": cleaned_final,
        "parsed": parsed,
        "parse_error": parse_error,
        "reasoning_has_leading_analysis": reasoning.lower().startswith("analysis"),
        "final_has_assistantfinal": "assistantfinal" in cleaned_final.lower().replace(" ", ""),
    }


def show_parse(raw_text: str, model_name: str):
    rec = parse_clean_output(raw_text, model_name=model_name)
    pprint(rec)
    if "gpt-oss" in model_name.lower():
        assert not rec["reasoning_has_leading_analysis"], "leading analysis was not removed"
        assert not rec["final_has_assistantfinal"], "assistantfinal leaked into final text"
    return rec


In [ ]:
gpt_oss_sample = """analysisWe need to send one natural message as hiring manager. Not formal offer. Should emphasize scope, growth, flexibility, impact, compensation. Use 1-3 sentences. JSON format. No mention of formal offer yet. assistantfinal{"Action":"DIALOGUE_MESSAGE","Message":"We’re excited about the possibility of you joining our platform team, and this role offers real ownership of a high-impact inference stack along with room to grow as the team scales. I’d love to answer any remaining questions and make sure the package and role feel like the right fit before we move forward."}"""

deepseek_sample = """<think>
We should keep the reply short and professional.
</think>
{"Action":"DIALOGUE_MESSAGE","Message":"I’m excited about the role and would love to talk through the package in a bit more detail."}
"""

print("GPT-OSS sample")
gpt_oss_result = show_parse(gpt_oss_sample, "openai/gpt-oss-20b")
print("\nDeepSeek sample")
deepseek_result = show_parse(deepseek_sample, "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B")


## Paste In Your Own Raw Output

Edit `MODEL_NAME` and `RAW_TEXT`, then rerun the next cell.

In [ ]:
MODEL_NAME = "openai/gpt-oss-20b"
RAW_TEXT = """analysisWe need to send one natural message as hiring manager. Not formal offer. assistantfinal{"Action":"DIALOGUE_MESSAGE","Message":"We’re excited about the role and would love to answer any remaining questions before we move forward."}"""

parsed = show_parse(RAW_TEXT, MODEL_NAME)


## Optional: Scan A JSONL File For Cleanup Issues

This is useful for mined outputs or examples files that contain `action_raw_text`, `_raw_text`, or `raw_text`.

In [ ]:
JSONL_PATH = None
MODEL_NAME = "openai/gpt-oss-20b"
LIMIT = 25
RAW_TEXT_FIELDS = ("action_raw_text", "_raw_text", "raw_text")


def iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def first_raw_text(row):
    for key in RAW_TEXT_FIELDS:
        value = row.get(key)
        if isinstance(value, str) and value.strip():
            return key, value
    action = row.get("action")
    if isinstance(action, dict):
        for key in RAW_TEXT_FIELDS:
            value = action.get(key)
            if isinstance(value, str) and value.strip():
                return f"action.{key}", value
    return None, None


def scan_jsonl(path, model_name=MODEL_NAME, limit=LIMIT):
    path = Path(path)
    rows = []
    for idx, row in enumerate(iter_jsonl(path)):
        key, raw_text = first_raw_text(row)
        if raw_text is None:
            continue
        parsed = parse_clean_output(raw_text, model_name=model_name)
        rows.append({
            "idx": idx,
            "record_id": row.get("record_id") or row.get("example_id") or row.get("conversation_id"),
            "raw_text_field": key,
            "reasoning_has_leading_analysis": parsed["reasoning_has_leading_analysis"],
            "final_has_assistantfinal": parsed["final_has_assistantfinal"],
            "reasoning_preview": parsed["reasoning"][:120],
            "final_preview": parsed["cleaned_final"][:120],
            "parsed": parsed["parsed"],
            "parse_error": parsed["parse_error"],
        })
        if len(rows) >= limit:
            break
    return rows


if JSONL_PATH:
    scan_results = scan_jsonl(JSONL_PATH)
    pprint(scan_results[:5])
    print(f"rows_scanned={len(scan_results)}")
else:
    print("Set JSONL_PATH to a file you want to inspect, then rerun this cell.")
